In [1]:
# import nifty.nifty.cl as ift
import nifty8 as ift
from phase_II.fast_wigner_function.utils import *
from scipy.interpolate import interp1d
from phase_II.utils.helpers import *

%matplotlib notebook

In [58]:
def interpolator(x_array, y_array):
    # if outside of bound, will be filled with 0's
    return interp1d(x_array, y_array, kind='cubic', bounds_error=False, fill_value=0)


def periodic_interpolator(x, y, kind='cubic'):
    dx = x[1] - x[0]                 # spacing
    P = x[-1] - x[0] + dx            # full period length
    # extend one period on both sides
    x_ext = np.concatenate([x - P, x, x + P])
    y_ext = np.tile(y, 3)
    return interp1d(x_ext, y_ext, kind=kind, bounds_error=True)


def Stress(harmonic_xi):
    # xi is assumed to be in standard hartley order
    xi_vals = harmonic_xi.val

    h = harmonic_xi.domain[0]
    f = h.get_k_length_array().val
    f = fftshift_to_hartley(hartley_to_fftshift(f, flip_negatives=True))  # hartley order: 0,... +f_nyq, a bit larger than -f_nyq, ..., almost 0

    k = f.copy()  # let the interpolator do the job
    N = len(f)

    # xi = interpolator(x_array=f, y_array=xi_vals)

    # sort only for interpolator
    idx_sort = np.argsort(f)
    xi = periodic_interpolator(f[idx_sort], xi_vals[idx_sort])

    f_cast, k_cast = (f[:, None], k[None, :])  # f are rows, k are columns
    Phi = xi(f_cast+1/2*k_cast).conj() * xi(f_cast-1/2*k_cast)

    Phi_field = ift.Field(ift.DomainTuple.make((h,h)), val=Phi)

    iFFT = ift.FFTOperator(domain=(h,h), space=1)  # apply accross columns

    S = iFFT(Phi_field)
    S_mat = S.val


    dk = k[1]-k[0]
    dt = 1.0 / (N * dk)
    t_dual = np.arange(N) * dt  # k-dual time might have different spacing.

    return S_mat, t_dual, f

In [51]:
L = 2
n_pix = 200
dt = L/n_pix

t = ift.RGSpace(shape=(n_pix,), distances=dt)
h = t.get_default_codomain()
h_dt = ift.DomainTuple.make(h)
f = h.get_k_length_array().val
f_hartley = fftshift_to_hartley(hartley_to_fftshift(f, flip_negatives=True))
f_fft = hartley_to_fftshift(f_hartley, flip_negatives=True)
t_values = np.arange(n_pix)*dt

In [52]:
harmonic_xi_dirac_delta = np.sqrt(L)*ift.from_random(h)

In [53]:
S_mat, t_dual, f = Stress(harmonic_xi_dirac_delta)

[  0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5 -99.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   

NameError: name 'stop' is not defined

In [6]:
visualize_stress_pcolormesh(S_mat.real, rows=f, cols=t_dual, sort_axes=True)
print(np.mean(S_mat.real), np.std(S_mat.real))

<IPython.core.display.Javascript object>

1.0066472519049576 34.49816944395854


In [7]:
num = 10
S_mat_collection = []
for _ in range(num):
    xi_power_dirac_delta = np.sqrt(L)*ift.from_random(h)
    S_mat, t_dual, f = Stress(xi_power_dirac_delta)
    S_mat_collection.append(S_mat)

In [8]:
S_mat_avg = np.mean(S_mat_collection, axis=0)

In [9]:
visualize_stress_pcolormesh(S_mat_avg.real, rows=f, cols=t_dual, sort_axes=True)
print(np.mean(S_mat_avg), np.std(S_mat_avg))

<IPython.core.display.Javascript object>

(1.008977939247307-6.468659422812297e-35j) 10.982058232134449


Below is a little ChatGpt program to test the complexity of the algorithm

In [10]:
import time
import numpy as np
import matplotlib.pyplot as plt

Ns = [200, 400, 800, 1600, 3200,] #10_000, 20_000]
runtimes = []

for N in Ns:
    L = 2
    dt = L / N
    t = ift.RGSpace(shape=(N,), distances=dt)
    h = t.get_default_codomain()
    xi_power_dirac_delta = np.sqrt(L) * ift.from_random(h)

    start = time.time()
    S_mat, t_dual, f = Stress(xi_power_dirac_delta)
    end = time.time()

    runtimes.append(end - start)
    print(f"N={N}, runtime={end-start:.3f}s")

# Use first runtime as reference
ref = runtimes[0]
N0 = Ns[0]

plt.figure()
plt.plot(Ns, runtimes, 'o-', label='measured runtime')

# reference scalings
plt.plot(Ns, [ref*( (N**3)/(N0**3) ) for N in Ns], '--', label=r"$\propto N^3$")
plt.plot(Ns, [ref*( (N**2)/(N0**2) ) for N in Ns], '--', label=r"$\propto N^2$")
plt.plot(Ns, [ref*( (N*np.log(N))/(N0*np.log(N0)) ) for N in Ns], '--', label=r"$\propto N\log N$")
plt.plot(Ns, [ref*( (N**2*np.log(N))/(N0**2*np.log(N0)) ) for N in Ns], '--', label=r"$\propto N^2\log N$")

plt.xlabel("N")
plt.ylabel("runtime [s]")
plt.legend()
plt.show()


N=200, runtime=0.003s
N=400, runtime=0.007s
N=800, runtime=0.025s
N=1600, runtime=0.101s
N=3200, runtime=0.421s


<IPython.core.display.Javascript object>

In [64]:
xi_peak = xi_field(case=1, N=n_pix, peak_frequency=-20, peak_amplitude=100, omegas=f_hartley)
xi_peak_field = ift.Field(h_dt, xi_peak)


Constructing xi field for case 1: Spike

	f star:  -20.0  at index  160


In [65]:
S_mat, t_dual, f = Stress(xi_peak_field)

In [66]:
visualize_stress_pcolormesh(S_mat.real, rows=f, cols=t_dual, sort_axes=True)
print(np.argwhere(S_mat.real==np.max(S_mat.real)))

<IPython.core.display.Javascript object>

[[160   0]]


In [55]:
t_star = 0.9
xi_peak_real = xi_field(case=1, N=n_pix, peak_frequency=t_star, peak_amplitude=100, omegas=t_values)
xi_peak_real_field = ift.Field(dt_(t), xi_peak_real)
print("\t Think of as in time domain! (So t star, not f star.)")
# fig = plt.figure()
# plt.plot(t_values, xi_peak_real_field.val)
# plt.show()


Constructing xi field for case 1: Spike

	f star:  0.9  at index  90
	 Think of as in time domain! (So t star, not f star.)


In [56]:
FFT = ift.FFTOperator(domain=t)
tilde_xi_peak_real_field = FFT(xi_peak_real_field)

# _ = plt.figure()
# plt.plot(f_hartley, tilde_xi_peak_real_field.val)
# plt.show()

In [59]:
S_mat, t_dual, f = Stress(tilde_xi_peak_real_field)

In [60]:
visualize_stress_pcolormesh(S_mat.real, rows=f, cols=t_dual, sort_axes=True)
print(np.argwhere(S_mat.real==np.max(S_mat.real)))

<IPython.core.display.Javascript object>

[[  3 110]
 [ 14 110]
 [ 15 110]
 [ 20 110]
 [ 28 110]
 [ 39 110]
 [ 40 110]
 [ 45 110]
 [ 53 110]
 [ 64 110]
 [ 65 110]
 [ 70 110]
 [ 78 110]
 [ 89 110]
 [ 90 110]
 [ 95 110]
 [103 110]
 [114 110]
 [115 110]
 [120 110]
 [128 110]
 [139 110]
 [140 110]
 [145 110]
 [153 110]
 [164 110]
 [165 110]
 [170 110]
 [178 110]
 [189 110]
 [190 110]
 [195 110]]


In [72]:
N = S_mat.shape[1]

# global max (use absolute to be safe)
rmax1, cmax1 = np.unravel_index(np.argmax(np.abs(S_mat)), S_mat.shape)
print("matrix max at row,col =", rmax1, cmax1)

dt_time = t_dual[1] - t_dual[0]  # spacing between consecutive columns

t_noncentered = cmax * dt_time
t_centered    = (cmax - N//2) * dt_time

print("t_noncentered =", t_noncentered)
print("t_centered    =", t_centered)



matrix max at row,col = [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]] [[  0   0   0 ... 199 199 199]
 [  0   1   2 ... 197 198 199]]
t_noncentered = 0.0
t_centered    = -1.0
